In [12]:
import sys
import os
sys.path.append('..')
os.chdir('..')

### Data Transformation

In [4]:
import os
%pwd

'c:\\Users\\HP\\Desktop\\Y3 S2\\projects\\Text-Summarizer\\research'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path

In [6]:
from src.textSummarizer.constants import *
from src.textSummarizer.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(self,
                config_path=CONFIG_FILE_PATH,
                params_filepath=PARAMS_FILE_PATH):
        self.config=read_yaml(config_path)
        self.paramss=read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self)-> DataTransformationConfig:
                config=self.config.data_transformation

                create_directories([config.root_dir])

                data_transformation_config=DataTransformationConfig(
                    root_dir=config.root_dir,
                    data_path=config.data_path,
                    tokenizer_name=config.tokenizer_name
                )

                return data_transformation_config


In [9]:
import os
from src.textSummarizer.config.configuration import logger
from transformers import AutoTokenizer
from datasets import load_from_disk   

### Data Transformation Component

In [16]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(config.tokenizer_name)

    def convert_examples_to_features(self,example_batch):
        input_encodings = self. tokenizer(example_batch['dialogue' ] , max_length = 1024, truncation = True )

        target_encodings = self. tokenizer(example_batch['summary' ], max_length = 128, truncation = True )

        return {
            'input_ids' : input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.convert_examples_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,'samsum_dataset'))

In [17]:
config=ConfigurationManager()
data_transformation_config=config.get_data_transformation_config()
data_transformation=DataTransformation(config=data_transformation_config)
data_transformation. convert()

[2026-03-23 14:29:33,794: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-03-23 14:29:33,797: INFO: common: yaml file: params.yaml loaded successfully]
[2026-03-23 14:29:33,798: INFO: common: created directory at: artifacts]
[2026-03-23 14:29:33,799: INFO: common: created directory at: artifacts/data_transformation]
[2026-03-23 14:29:34,161: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-23 14:29:34,206: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-03-23 14:29:34,473: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-03-23 14:29:34,522: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-c

Saving the dataset (1/1 shards): 100%|██████████| 819/819 [00:00<00:00, 43332.98 examples/s]
